In [1]:
1+1

2

In [2]:
#%pip install torchvision tqdm 

In [5]:
# Attention 
import torch

sentence = "this is a sample sentence for embedding"
sentence2 = "this is sentence embedding"

dc  = {token: i for i,token in enumerate(sentence.replace(",", "").split())}

dc

{'this': 0,
 'is': 1,
 'a': 2,
 'sample': 3,
 'sentence': 4,
 'for': 5,
 'embedding': 6}

In [6]:
vocab_size =  len(dc) 
emb = torch.nn.Embedding(vocab_size, 3) # trainierbare Parameter , 768 - 4096 
emb.weight.data # Zufallsinitialisierte Embedding-Matrix

tensor([[ 0.1386,  0.2013,  0.6050],
        [-1.5719,  1.2736,  0.0712],
        [-1.4202,  0.4589,  1.5002],
        [-0.0087,  0.0191, -0.8390],
        [-0.7362,  1.0581, -1.1556],
        [ 0.3277,  0.9002,  0.6397],
        [ 0.5253,  0.7625, -0.5026]])

In [19]:
sentence_int = torch.tensor([dc[token] for token in sentence.split()])
sentence_emb = emb(sentence_int)
sentence_emb

tensor([[ 0.1386,  0.2013,  0.6050],
        [-1.5719,  1.2736,  0.0712],
        [-1.4202,  0.4589,  1.5002],
        [-0.0087,  0.0191, -0.8390],
        [-0.7362,  1.0581, -1.1556],
        [ 0.3277,  0.9002,  0.6397],
        [ 0.5253,  0.7625, -0.5026]], grad_fn=<EmbeddingBackward0>)

In [20]:
# Q, K , V

d = sentence_emb.shape[1]
d_Q, d_K, d_V = 2, 2, 4

W_Q = torch.nn.Parameter(torch.rand(d, d_Q))
W_K = torch.nn.Parameter(torch.rand(d, d_K))
W_V = torch.nn.Parameter(torch.rand(d, d_V))

W_V

Parameter containing:
tensor([[0.5616, 0.0525, 0.2303, 0.4966],
        [0.6519, 0.4878, 0.1120, 0.5785],
        [0.8699, 0.4378, 0.8834, 0.2665]], requires_grad=True)

In [21]:
queries = sentence_emb @ W_Q
keys = sentence_emb @ W_K
values = sentence_emb @ W_V

In [22]:
scores_raw = queries @ keys.T # QK^T für alle Tokens/Relevanzmaß
scores_raw

tensor([[ 0.4167, -0.1294,  0.2893, -0.3880, -0.4395,  0.7779,  0.2736],
        [-0.2072,  0.2001,  0.4122,  0.0516, -0.0453, -0.6377, -0.5976],
        [-0.1525,  0.1387,  0.2685,  0.0469, -0.0168, -0.4536, -0.4109],
        [-0.2689,  0.0862, -0.1756,  0.2476,  0.2784, -0.5070, -0.1858],
        [-0.1959,  0.1457,  0.2114,  0.0942,  0.0418, -0.5226, -0.4171],
        [ 0.9933, -0.2896,  0.7675, -0.9448, -1.0846,  1.8193,  0.5875],
        [ 0.6617, -0.2007,  0.4793, -0.6213, -0.7074,  1.2264,  0.4180]],
       grad_fn=<MmBackward0>)

In [23]:
scores = scores_raw/math.sqrt(d_K) # Skalierung 
attn_weights = torch.nn.functional.softmax(scores, dim = 1) # Softmax über die Zeilen
attn_weights # Aufmerksamkeitsgewichte

tensor([[0.1695, 0.1152, 0.1549, 0.0959, 0.0925, 0.2188, 0.1532],
        [0.1298, 0.1732, 0.2012, 0.1559, 0.1456, 0.0958, 0.0985],
        [0.1339, 0.1645, 0.1803, 0.1542, 0.1474, 0.1082, 0.1115],
        [0.1224, 0.1573, 0.1307, 0.1763, 0.1802, 0.1034, 0.1298],
        [0.1304, 0.1661, 0.1740, 0.1601, 0.1543, 0.1035, 0.1115],
        [0.1892, 0.0764, 0.1613, 0.0481, 0.0435, 0.3394, 0.1420],
        [0.1806, 0.0981, 0.1587, 0.0729, 0.0686, 0.2692, 0.1520]],
       grad_fn=<SoftmaxBackward0>)

In [24]:
attn_output = attn_weights @ values # Attention(Q, K, V) = softmax(QK^T/sqrt(d_k))V 
attn_output # Kontextvektor 

tensor([[ 4.5872e-01,  4.0485e-01,  1.9993e-01,  2.9705e-01],
        [ 2.0273e-01,  3.3711e-01,  3.6523e-02,  1.2764e-01],
        [ 2.0990e-01,  3.2893e-01,  2.3883e-02,  1.4839e-01],
        [ 1.2151e-01,  2.7142e-01, -9.3316e-02,  1.4598e-01],
        [ 1.8668e-01,  3.1765e-01, -3.8653e-04,  1.4169e-01],
        [ 7.0446e-01,  5.0059e-01,  4.0441e-01,  4.1465e-01],
        [ 5.7033e-01,  4.4813e-01,  2.9351e-01,  3.4997e-01]],
       grad_fn=<MmBackward0>)

In [25]:
import math
from pathlib import Path
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm


batch_size = 128
num_workers = 4

data_dir = Path("hands-on/data")
data_dir.mkdir(parents=True, exist_ok=True)


cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)
normalize = transforms.Normalize(mean=cifar_mean, std=cifar_std)

train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])


train_ds = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=train_transforms)
test_ds = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=test_transforms)

device = "cpu"
pin_memory = device == "cuda"
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

class_names = train_ds.classes
print(f"Train samples: {len(train_ds)}, Test samples: {len(test_ds)}")



Train samples: 50000, Test samples: 10000


In [28]:
# https://arxiv.org/pdf/2010.11929

class PatchEmbedding(nn.Module):
    def __init__(self, img_size = 32, patch_size = 4, in_channels = 3, embed_dim = 128 ): # 
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride= patch_size) # hier passiert das Schneiden der Patches
    
    def forward(self,x):
        x = self.proj(x) # (B, embed_dim, img_size/patch_size, img_size/patch_size )
        x = x.flatten(2).transpose(1,2) # (B, num_patches, embed_dim)
        return x 



class MLP(nn.Module):
    def __init__(self,embed_dim = 128 , mlp_ratio = 4.0 , drop = 0.0):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, embed_dim*mlp_ratio) 
        self.act = nn.GELU() # für viele Transformer empfohlen 
        self.fc2 = nn.Linear(embed_dim*mlp_ratio, embed_dim) 
        self.drop = nn.Dropout(drop) 

    def forward(self,x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x 



class Block(nn.Module):
    def __init__(self,embed_dim = 128, num_heads=4, mlp_ratio = 4.0 , drop = 0.0, attn_drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout = attn_drop, batch_first=True)
        self.drop = nn.Dropout(drop) 
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_ratio, drop) 

    def forward(self,x):
        norm1 = self.norm1(x)
        attn_out, _ = self.attn(norm1, norm1, norm1)
        x = x + self.drop(attn_out) # unterer Teil done 
        norm2 = self.norm2(x) 
        x = x + self.drop(self.mlp(norm2)) # oberer Teil
        return x 
        

        
class VisionTransformer(nn.Module):
    def __init__(self, 
                 img_size =32,  
                 patch_size = 4, 
                 in_channels = 3, 
                 num_classes = 10, 
                 embed_dim = 128, 
                 depth=6, # Tiefe 
                num_heads = 4, 
                mlp_ratio = 4,
                drop = 0.1,
                attn_drop = 0.0
                ):
        super().__init__()
        # PatchEmbedding 

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim) # in_channels = 3 weil RGB
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.randn(1,1, embed_dim)) # [class] oder [CLS] 
        self.pos_embed = nn.Parameter(torch.randn(1,num_patches + 1, embed_dim)) # Positionsembeddings = Kontextlänge 
        self.drop = nn.Dropout(drop)
        
        # TransformerBlock 
        self.transformer = nn.ModuleList(
            [
                Block(embed_dim , num_heads, mlp_ratio , drop , attn_drop) for _ in range(depth) # Tiefe 6
            ]
        )

        self.norm = nn.LayerNorm(embed_dim)
        # MLP Head
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self,x):
        B = x.shape[0] # Batch-Größe
        x = self.patch_embed(x) 
        cls_tokens = self.cls_token.expand(B, -1, -1) # Batchweise 
        x = torch.cat((cls_tokens,x), dim = 1) 
        x = x + self.pos_embed 
        x = self.drop(x) 

        for block in self.transformer:
            x = block(x) 
        x = self.norm(x)
        cls_out = x[:,0]
        return self.head(cls_out) 
        

In [29]:
model = VisionTransformer(
                 img_size =32,  
                 patch_size = 4, 
                 in_channels = 3, 
                 num_classes = 10, 
                 embed_dim = 128, 
                 depth=6, # Tiefe 
                num_heads = 4, 
                mlp_ratio = 4,
                drop = 0.1,
                attn_drop = 0.0
                )
model.to(torch.device(device))

In [30]:
lr = 3e-4
epochs = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {num_params/1e6:.2f}M")


Model params: 1.21M


In [31]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient Clipping 
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

@torch.inference_mode()
def evaluate(model, dataloader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


In [ ]:
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, device)
    scheduler.step()
    print(f"Epoch {epoch + 1}/{epochs} | train loss {train_loss:.4f} acc {train_acc:.3f} | val loss {val_loss:.4f} acc {val_acc:.3f}")



Epoch 1/10 | train loss 1.8499 acc 0.320 | val loss 1.5713 acc 0.432


Epoch 2/10 | train loss 1.5852 acc 0.425 | val loss 1.4295 acc 0.492


Epoch 3/10 | train loss 1.4743 acc 0.463 | val loss 1.3490 acc 0.518


Epoch 4/10 | train loss 1.4026 acc 0.493 | val loss 1.2809 acc 0.544


Epoch 5/10 | train loss 1.3466 acc 0.512 | val loss 1.2715 acc 0.547


 64%|█████████████████████████████████████████████▏                         | 249/391 [10:40<08:50,  3.73s/it]